# Autoresearch Experiment Analysis

Analysis of autonomous marketing-campaign recommender experiments from `results.tsv`.

The metric is **`policy_value`** — expected net profit ($) per targeted customer. **Higher is better.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# results.tsv columns: commit, policy_value, regret, status, description
df = pd.read_csv("results.tsv", sep="\t")
df["policy_value"] = pd.to_numeric(df["policy_value"], errors="coerce")
df["regret"] = pd.to_numeric(df["regret"], errors="coerce")
df["status"] = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep = counts.get("KEEP", 0)
n_discard = counts.get("DISCARD", 0)
n_crash = counts.get("CRASH", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  policy_value={row['policy_value']:.6f}  regret={row['regret']:.6f}  {row['description']}")

## Policy value over time

Track how the best (kept) `policy_value` climbs as experiments progress. The running maximum is the "frontier" — the best profit-per-customer achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
baseline = valid.loc[0, "policy_value"]

# Discarded experiments as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["policy_value"], c="#cccccc", s=14, alpha=0.6, zorder=2, label="Discarded")

# Kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["policy_value"], c="#2ca02c", s=40, zorder=3, label="Kept")

# Running-max frontier
frontier = valid["policy_value"].cummax()
ax.plot(valid.index, frontier, c="#1f77b4", lw=2, zorder=4, label="Best so far")

ax.axhline(baseline, c="#888", ls="--", lw=1, label=f"Baseline ({baseline:.4f})")
ax.set_xlabel("Experiment #")
ax.set_ylabel("policy_value ($ / customer)  — higher is better")
ax.set_title("Autoresearch: policy value over experiments")
ax.legend()
plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
baseline = df.iloc[0]["policy_value"]
best = kept["policy_value"].max()
best_row = kept.loc[kept["policy_value"].idxmax()]

print(f"Baseline policy_value: {baseline:.6f}")
print(f"Best policy_value:     {best:.6f}")
print(f"Total improvement:     +{best - baseline:.6f} ({(best - baseline) / baseline * 100:.2f}%)")
print(f"Best experiment:       {best_row['description']}")
print(f"Best regret:           {best_row['regret']:.6f}  (lower = closer to the oracle)")

## Top hits (kept experiments by improvement)

Each kept experiment's delta is measured vs the previous kept state (experiments are cumulative).

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev"] = kept["policy_value"].shift(1)
kept["delta"] = kept["policy_value"] - kept["prev"]
hits = kept.iloc[1:].sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>9}  {'policy_value':>12}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+9.6f}  {row['policy_value']:12.6f}  {row['description']}")

## Inspect the current recommender

Fit whatever `recommend.py` currently contains and look at its metrics and the mix of campaigns it recommends. (Run `uv run prepare.py` first if the dataset isn't generated yet.)

In [ ]:
from prepare import CAMPAIGN_NAMES, load_train_experiment, evaluate_policy
from recommend import Recommender

rec = Recommender().fit(load_train_experiment())
m = evaluate_policy(rec)

print(f"policy_value : {m['policy_value']:.4f}   (oracle {m['oracle_value']:.4f}, "
      f"best-single {m['best_single_value']:.4f}, no-contact {m['no_contact_value']:.4f})")
print(f"regret       : {m['regret']:.4f}")
print(f"gain captured: {100 * m['gain_captured_frac']:.1f}% of the personalization gain\n")

counts = np.array(m["action_counts"])
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(CAMPAIGN_NAMES, 100 * counts / counts.sum(), color="#2ca02c")
ax.set_ylabel("% of customers")
ax.set_title("Recommended campaign mix")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()